In [1]:
from Data_query.trino_config import *
import numpy as np
import boto3
import random
import pytz
session = boto3.Session()
s3 = boto3.client('s3')

In [5]:
stop_trino()

Trino service stopping triggered.


In [2]:
big_workers = 1
workers = 0
num_workers = max(workers, big_workers)
ensure_trino_running(worker_desired_count = workers, big_worker_desired_count=big_workers)
sleep(60)  # wait for trino to be ready

Trino service is not running. Starting the service...
Trino service triggered.
Service trino-service is now stable.


In [3]:
v1=207
v2=220
v3=240
v4=258
voltwatt_V = 253
Q1=.44
Q4=.60
thr1 = -0.1
thr2 = 0.1
thr3 = .9
thr4 = 1.1
def run_func(args):
    year, month, split_cons = args
    df = iceberg_sql(f"""
                with 
                data as (
                    select site_id, t_stamp, sum(power*circuit_polarity/1000) as P_kW, 
                        sum(energy_reactive*circuit_polarity/1000*12) as Q_kvar, 
                            avg(voltage) as V, max(ac_capacity_kw) as ac_capacity_kw, max(S_99) as S_99
                    from 
                        (select circuit_id, t_stamp, power, energy_reactive, voltage 
                            from ts where year={year} and month={month} and is_pv=True and voltage > 0 and voltage < 300 and {split_cons} ) as ts1
                    inner join (select distinct site_id, circuit_id, circuit_polarity, ac_capacity_kw, S_99 from meta_up23c where is_pv = True) as m on ts1.circuit_id = m.circuit_id
                    group by site_id, t_stamp),
                pq as (
                    select site_id, t_stamp, P_kW, Q_kvar, V, ac_capacity_kw, S_99,
                    case when V < {v1} then {Q1}*ac_capacity_kw 
                    when V < {v2} then ({Q1}* ac_capacity_kw - 0) / ({v1} - {v2}) * (V - {v2})
                    when V <= {v3} then 0
                    when V < {v4} then -(0 - {Q4}* ac_capacity_kw) / ({v3} - {v4}) * (V - {v4}) - {Q4}* ac_capacity_kw
                    else - {Q4}* ac_capacity_kw end as Q_voltvar,
                    case when abs(P_kW) < .2 * ac_capacity_kw then 0
                    when abs(P_kW) <= .6 * ac_capacity_kw then - 0.44 * ac_capacity_kw
                    when abs(P_kW) <= .8 * ac_capacity_kw then 
                        case when  (power(abs(P_kW)/0.8, 2) - power(abs(P_kW), 2) ) < 0 then 0
                        else -sqrt(power(abs(P_kW)/0.8, 2) - power(abs(P_kW), 2)) end
                    else 
                        case when ( power(ac_capacity_kw, 2) - power(abs(P_kW), 2) ) < 0 then 0
                        else -sqrt( power(ac_capacity_kw, 2) - power(abs(P_kW), 2) ) end
                    end as Q_cap_absorbing
                    from data
                    ),
                pq2 as (
                    select site_id, t_stamp, P_kW, Q_kvar, V, ac_capacity_kw, S_99, Q_voltvar, Q_cap_absorbing,
                    -Q_cap_absorbing as Q_cap_supplying,
                    Q_voltvar + .04 * ac_capacity_kw as Q_voltvar_max,
                        Q_voltvar - .04 * ac_capacity_kw as Q_voltvar_min
                    from pq
                    ),
                pq3 as (
                    select site_id, t_stamp, P_kW, Q_kvar, V, ac_capacity_kw, S_99, Q_voltvar, Q_cap_absorbing, Q_cap_supplying, Q_voltvar_max, Q_voltvar_min,
                    case when Q_voltvar_max < 0 then greatest(Q_voltvar_max, Q_cap_absorbing + .04*ac_capacity_kw) else Q_voltvar_max end as Q_voltvar_max_final,
                    case when Q_voltvar_min > 0 then least(Q_voltvar_min, Q_cap_supplying - .04*ac_capacity_kw) else Q_voltvar_min end as Q_voltvar_min_final
                    from pq2
                ),
                pq4 as (
                            select site_id, t_stamp, P_kW, Q_kvar, V, ac_capacity_kw, S_99, Q_voltvar_max_final, Q_voltvar_min_final, 
                            CASE 
                                WHEN abs(Q_kvar) / (abs(Q_voltvar_max_final) + 1e-9) 
                                    <= abs(Q_kvar) / (abs(Q_voltvar_min_final) + 1e-9)
                                THEN 
                                    CASE 
                                        WHEN Q_voltvar_max_final + Q_voltvar_min_final = 0 THEN 1
                                        ELSE sign(Q_voltvar_max_final) * sign(Q_kvar)
                                    END 
                                    * (abs(Q_kvar) / (abs(Q_voltvar_max_final) + 1e-9))
                                ELSE 
                                    CASE 
                                        WHEN Q_voltvar_max_final + Q_voltvar_min_final = 0 THEN 1
                                        ELSE sign(Q_voltvar_min_final) * sign(Q_kvar)
                                    END 
                                    * (abs(Q_kvar) / (abs(Q_voltvar_min_final) + 1e-9))
                                END AS Q_impact
                        from pq3),
                    pq5 as (
                        select pq4.site_id, pq4.t_stamp, P_kW, Q_kvar, sqrt(power(Q_kvar, 2) + power(P_kW, 2)) as S_kVA,
                        V, ac_capacity_kw, S_99, Q_voltvar_max_final, Q_voltvar_min_final, Q_impact,
                        case when (Q_kvar < Q_voltvar_min_final or Q_kvar > Q_voltvar_max_final) and Q_impact < {thr1} then 
                        least(abs(Q_kvar - Q_voltvar_min_final), abs(Q_kvar - Q_voltvar_max_final)) else 0  end as Q_adverse,
                        case when (Q_kvar < Q_voltvar_min_final or Q_kvar > Q_voltvar_max_final) and (Q_impact >= {thr1} and Q_impact <= {thr2}) then 
                        least(abs(Q_kvar - Q_voltvar_min_final), abs(Q_kvar - Q_voltvar_max_final)) else 0 end as Q_inactive,
                        case when (Q_kvar < Q_voltvar_min_final or Q_kvar > Q_voltvar_max_final) and (Q_impact > {thr2} and Q_impact < {thr3}) then 
                        least(abs(Q_kvar - Q_voltvar_min_final), abs(Q_kvar - Q_voltvar_max_final)) else 0  end as Q_minor_deviation,
                        case when (Q_kvar < Q_voltvar_min_final or Q_kvar > Q_voltvar_max_final) and (Q_impact >= {thr3} and Q_impact <= {thr4}) then 
                        least(abs(Q_kvar - Q_voltvar_min_final), abs(Q_kvar - Q_voltvar_max_final)) else 0  end as Q_major_deficit,
                        case when (Q_kvar < Q_voltvar_min_final or Q_kvar > Q_voltvar_max_final) and (Q_impact > {thr4}) then 
                        least(abs(Q_kvar - Q_voltvar_min_final), abs(Q_kvar - Q_voltvar_max_final)) else 0  end as Q_major_surplus,
                        case when V <= {voltwatt_V} and sqrt(power(Q_kvar, 2) + power(P_kW, 2)) >= S_99 then uncurtailed_P - P_kW end as curtailment_voltvar
                        from pq4 left join all_uncurtailedPV as a on pq4.site_id = a.site_id and pq4.t_stamp = a.t_stamp
                    ),
                    pq6 as (
                        select site_id, t_stamp, day(t_stamp) as day, month(t_stamp) as month, year(t_stamp) as year,
                        case when (hour(t_stamp) >= 20 or hour(t_stamp) <= 7) then 'day' else 'night' end as day_night,
                        Q_adverse, Q_inactive, Q_minor_deviation, Q_major_deficit, Q_major_surplus, 
                        Q_adverse + Q_inactive + Q_minor_deviation + Q_major_deficit + Q_major_surplus as nonconformance_voltvar,
                        curtailment_voltvar
                        from pq5
                    )
                select year, month, day, day_night, site_id, 
                sum(nonconformance_voltvar) as nonconformance_voltvar_sum,
                sum(Q_adverse) as Q_adverse_sum,
                sum(Q_inactive) as Q_inactive_sum,
                sum(Q_minor_deviation) as Q_minor_deviation_sum,
                sum(Q_major_deficit) as Q_major_deficit_sum,
                sum(Q_major_surplus) as Q_major_surplus_sum,
                sum(curtailment_voltvar) as curtailment_voltvar_sum,
                sum(case when nonconformance_voltvar > 0 then 1 else 0 end) as nonconformance_voltvar_count,
                sum(case when Q_adverse > 0 then 1 else 0 end) as Q_adverse_count,
                sum(case when Q_inactive > 0 then 1 else 0 end) as Q_inactive_count,
                sum(case when Q_minor_deviation > 0 then 1 else 0 end) as Q_minor_deviation_count,
                sum(case when Q_major_deficit > 0 then 1 else 0 end) as Q_major_deficit_count,
                sum(case when Q_major_surplus > 0 then 1 else 0 end) as Q_major_surplus_count,
                sum(case when curtailment_voltvar > 0 then 1 else 0 end) as curtailment_voltvar_count,
                count(nonconformance_voltvar) as total_count
                from pq6
                group by year, month, day, day_night, site_id
                """)
    # sleep(20)
    print(f"Completed year={year}, month={month},  {split_cons.replace('system.bucket(postcode, 16)', 'bucket')}")
    return df

tasks = [(year, month, split_cons) for year in (2024, ) for month in range(1, 2) 
         for split_cons in ['system.bucket(postcode, 16) <= 3', '(system.bucket(postcode, 16) > 3 and system.bucket(postcode, 16) <= 7)', 
                            '(system.bucket(postcode, 16) > 7 and system.bucket(postcode, 16) <= 11)', 'system.bucket(postcode, 16) > 11'] ]
df = trino_parallel(run_func, tasks, num_workers=num_workers)

Completed year=2024, month=1,  bucket <= 3
Completed year=2024, month=1,  (bucket > 3 and bucket <= 7)
Completed year=2024, month=1,  (bucket > 7 and bucket <= 11)
Completed year=2024, month=1,  bucket > 11
Combining results from all tasks.


In [4]:
df.sort_values('curtailment_voltvar_sum', ascending=False)

,year,month,day,day_night,site_id,nonconformance_voltvar_sum,Q_adverse_sum,Q_inactive_sum,Q_minor_deviation_sum,Q_major_deficit_sum,Q_major_surplus_sum,curtailment_voltvar_sum,nonconformance_voltvar_count,Q_adverse_count,Q_inactive_count,Q_minor_deviation_count,Q_major_deficit_count,Q_major_surplus_count,curtailment_voltvar_count,total_count
367026,2024,1,8,day,929712738,0.785689,0.000000,0.000000,0.000000,0.000000,0.785689,4.825196e+00,3,0,0,0,0,3,1,144
8399,2024,1,4,day,137085889,161.021292,0.000000,12.967308,121.512756,0.000000,26.541228,1.362540e+00,124,0,11,109,0,4,1,144
181186,2024,1,2,day,1297035097,48.553637,0.000000,0.000000,36.430143,0.067320,12.056175,8.527528e-01,48,0,0,40,2,6,1,144
389588,2024,1,29,day,1245111907,40.738369,26.557154,0.000000,0.000000,0.038051,14.143164,1.578025e-01,26,16,0,0,2,8,1,143
572247,2024,1,18,day,616600992,72.870571,0.000000,0.000000,72.870571,0.000000,0.000000,8.526513e-14,104,0,0,104,0,0,6,144
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683410,2024,1,27,night,1065828100,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0,0,0,0,0,0,0,144
683411,2024,1,28,night,1065828100,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0,0,0,0,0,0,0,144
683412,2024,1,29,night,1065828100,0.086183,0.000000,0.046093,0.000000,0.040090,0.000000,NaN,3,0,2,0,1,0,0,144
683413,2024,1,30,night,1065828100,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0,0,0,0,0,0,0,144


In [ ]:
# 13 timestamps with conformant: P < threshold and 6 uncurtailed > threshold, 7  

In [10]:
71-58

13

In [12]:
iceberg_sql("""
select * from meta_up23c where  site_id = 250484079 limit 10
""")

,site_id,state,postcode,longitude,latitude,dnsp_name,dc_capacity_kw,ac_capacity_kw,export_limit_kw,monitoring_start,...,m_id,avg_pf,std_pf,pf_99,pf_01,n_long,n_lat,distance_km,s_99,flex_export_detected
0,250484079,ACT,2900,149.05,-35.415,Actewagl,60.84,50.0,None,2022-09-01,...,M27,0.999732,0.000081,0.999876,0.999551,149.06,-35.42,1.241018,49.386868,False
1,250484079,ACT,2900,149.05,-35.415,Actewagl,60.84,50.0,None,2022-09-01,...,M27,0.999732,0.000081,0.999876,0.999551,149.06,-35.42,1.241018,49.386868,False
2,250484079,ACT,2900,149.05,-35.415,Actewagl,60.84,50.0,None,2022-09-01,...,M27,0.999732,0.000081,0.999876,0.999551,149.06,-35.42,1.241018,49.386868,False
